---
# Topic 2: Modules, Packages & Virtual Environments

A **module** is any Python file. A **package** is a folder of modules with an `__init__.py` file. The entire Python ecosystem — NumPy, Pandas, PyTorch, scikit-learn — is a collection of packages. Understanding modules is understanding how Python's ecosystem is structured.

---

## 2.1 Importing Modules — All Forms

### 📊 [VISUAL] Import Forms

```
FORMS OF IMPORT:

import module                  -> module.function()      full name required
import module as alias         -> alias.function()       shorter name
from module import name        -> function()             direct access
from module import name as x  -> x()                    aliased direct
from module import *           -> function()             AVOID — pollutes namespace

EXAMPLES:
import numpy                   -> numpy.array([1,2,3])
import numpy as np             -> np.array([1,2,3])       ← STANDARD
from numpy import array        -> array([1,2,3])
from numpy import array as arr -> arr([1,2,3])
```

In [ ]:
# ── math module
import math
print(math.pi)           # 3.14159...
print(math.sqrt(144))    # 12.0
print(math.ceil(3.2))    # 4
print(math.floor(3.9))   # 3
print(math.log(math.e))  # 1.0

# ── random module (used in ML for shuffling, sampling)
import random
random.seed(42)                      # reproducibility — always set in ML!
print(random.random())               # float in [0,1)
print(random.randint(1, 10))         # int in [1,10]
data = [1,2,3,4,5,6,7,8,9,10]
random.shuffle(data)                 # in-place shuffle
sample = random.sample(data, 3)      # 3 unique random items
print("Shuffled:", data)
print("Sample:", sample)

# ── os module (file paths, environment)
import os
print("CWD:", os.getcwd())
os.makedirs('output/models', exist_ok=True)  # create nested dirs safely
print("Path join:", os.path.join('data', 'train', 'images'))

# ── pathlib (modern, preferred over os.path)
from pathlib import Path
p = Path('data') / 'train' / 'images'   # path joining with /
print("pathlib path:", p)
print("Exists:", p.exists())

## 2.2 Creating Your Own Modules

Any `.py` file is a module. You can import it from other files in the same directory. This is how you split large projects into manageable, reusable pieces.

### 📊 [VISUAL] Professional ML Project Folder Structure

```
my_ml_project/
├── main.py                <- entry point, runs the experiment
├── config.py              <- hyperparameters and settings
├── data/
│   ├── __init__.py        <- makes 'data' a package
│   ├── loader.py          <- dataset loading logic
│   └── preprocessor.py    <- cleaning, normalisation
├── models/
│   ├── __init__.py
│   ├── linear.py          <- LinearRegression class
│   └── neural.py          <- NeuralNetwork class
├── utils/
│   ├── __init__.py
│   ├── metrics.py         <- accuracy, f1, confusion matrix
│   └── visualiser.py      <- plotting helpers
├── notebooks/
│   └── experiments.ipynb  <- Jupyter exploration
├── requirements.txt       <- list of pip dependencies
└── README.md
```

The code below simulates what `utils/metrics.py` and `main.py` would look like:

In [ ]:
# ── Simulating utils/metrics.py ─────────────────────────────────────────────
"""Evaluation metrics for classification tasks."""

def accuracy(y_true, y_pred):
    """Fraction of correct predictions."""
    if len(y_true) != len(y_pred):
        raise ValueError('y_true and y_pred must have same length')
    correct = sum(yt == yp for yt, yp in zip(y_true, y_pred))
    return correct / len(y_true)

def precision(y_true, y_pred, positive=1):
    """True positives / (True positives + False positives)."""
    tp = sum(yt==positive and yp==positive for yt,yp in zip(y_true,y_pred))
    fp = sum(yt!=positive and yp==positive for yt,yp in zip(y_true,y_pred))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(y_true, y_pred, positive=1):
    """True positives / (True positives + False negatives)."""
    tp = sum(yt==positive and yp==positive for yt,yp in zip(y_true,y_pred))
    fn = sum(yt==positive and yp!=positive for yt,yp in zip(y_true,y_pred))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


# ── Simulating main.py ───────────────────────────────────────────────────────
y_true = [1, 0, 1, 1, 0, 1]
y_pred = [1, 0, 0, 1, 0, 1]

print(f'Accuracy:  {accuracy(y_true, y_pred):.2%}')   # 83.33%
print(f'Precision: {precision(y_true, y_pred):.2%}')  # 100.00%
print(f'Recall:    {recall(y_true, y_pred):.2%}')     # 75.00%

## 2.3 The `__name__ == '__main__'` Guard

Every Python file has a built-in variable `__name__`. When a file is **run directly**, `__name__` equals `'__main__'`. When it is **imported as a module**, `__name__` equals the file name. This guard lets you write code that only runs when the file is executed directly.

In [ ]:
# ── Simulating metrics.py with the guard ─────────────────────────────────────

def accuracy(y_true, y_pred):
    return sum(yt==yp for yt,yp in zip(y_true, y_pred)) / len(y_true)


if __name__ == '__main__':
    # This block ONLY runs when you execute: python metrics.py
    # It does NOT run when you do: from utils.metrics import accuracy
    print('Running metrics.py tests...')
    y_t = [1, 0, 1, 1, 0]
    y_p = [1, 0, 0, 1, 0]
    print(f'Test accuracy: {accuracy(y_t, y_p):.2%}')   # 80.00%
    print('All tests passed.')

# In a notebook __name__ is '__main__', so the block WILL run here:
print(f"Current __name__: {__name__}")

## 2.4 Virtual Environments and pip

A virtual environment is an **isolated Python installation** for a specific project. Industry standard — every professional ML project uses one.

### 📊 [VISUAL] Why Virtual Environments Matter

```
WITHOUT venv (dangerous):         WITH venv (correct):

Global Python                      Global Python
  numpy==1.24  (project A needs)   ├── venv_project_a/
  numpy==1.19  (project B needs)   │     numpy==1.24  (isolated)
  CONFLICT! One must break.        └── venv_project_b/
                                         numpy==1.19  (isolated)
                                   No conflict. Both work perfectly.
```

The commands below are **terminal commands** — run them outside Jupyter in your terminal or Anaconda prompt:

In [ ]:
# ── TERMINAL COMMANDS — run these in your terminal, NOT in Jupyter ────────────

# Create a virtual environment
# python -m venv my_ml_env

# Activate (Windows)
# my_ml_env\Scripts\activate

# Activate (Mac/Linux)
# source my_ml_env/bin/activate

# Install packages
# pip install numpy pandas scikit-learn matplotlib

# Save dependencies
# pip freeze > requirements.txt

# Install from requirements on another machine
# pip install -r requirements.txt

# Deactivate
# deactivate

# ── CONDA (Anaconda — preferred for ML/data science) ─────────────────────────
# conda create -n ml_env python=3.10
# conda activate ml_env
# conda install numpy pandas scikit-learn
# conda deactivate

print("Virtual environment commands printed above — run them in your terminal.")

## ✏️ Exercises — Modules & Packages

**[EXERCISE 4 — Medium]** Create a proper Python package: a folder called `mathutils` with `__init__.py`. Inside, create two modules: `stats.py` (functions: `mean`, `median`, `std_dev`) and `vector.py` (functions: `dot_product`, `cosine_similarity`). In `main.py`, import and use both.